In [ ]:
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

!pip install evaluate
import evaluate

In [ ]:
!pip install pandas

In [ ]:
import requests

url = "https://raw.githubusercontent.com/teropa/nlp/master/resources/corpora/conll2000/train.txt"

text = requests.get(url).text
lines = text.split("\n")

In [ ]:
sentences = []
pos_labels = []
chunk_labels = []

sentence = []
pos = []
chunk = []

for line in lines:
    line = line.strip()

    if line == "":
        if sentence:
            sentences.append(sentence)
            pos_labels.append(pos)
            chunk_labels.append(chunk)
            sentence, pos, chunk = [], [], []
    else:
        parts = line.split()
        word = parts[0]
        p = parts[1]
        c = parts[2]

        sentence.append(word)
        pos.append(p)
        chunk.append(c)

if sentence:
    sentences.append(sentence)
    pos_labels.append(pos)
    chunk_labels.append(chunk)

print("Total Sentences:", len(sentences))


In [ ]:
TASK = "chunk"   # 👉 change to "pos" or "chunk"

if TASK == "pos":
    labels = pos_labels
else:
    labels = chunk_labels

In [ ]:
# ==============================
# STEP 3: CREATE DATASET
# ==============================

dataset = Dataset.from_dict({
    "tokens": sentences,
    "labels": labels
})


In [ ]:
# ==============================
# STEP 4: LABEL MAPPING
# ==============================

unique_labels = sorted(list(set(tag for sent in labels for tag in sent)))

label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}


In [ ]:
# ==============================
# STEP 5: TOKENIZER
# ==============================

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# ==============================
# STEP 6: TOKENIZATION + ALIGNMENT
# ==============================

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True,
        max_length=128
    )

    new_labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []

        prev = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            prev = word_idx

        new_labels.append(label_ids)

    tokenized["labels"] = new_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)


In [ ]:
# ==============================
# STEP 7: TRAIN / VAL SPLIT
# ==============================

train_size = int(0.8 * len(tokenized_dataset))

train_dataset = tokenized_dataset.select(range(train_size))
val_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

In [ ]:
# ==============================
# STEP 8: MODEL
# ==============================

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# ==============================
# STEP 9: METRICS
# ==============================

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    total = 0
    correct = 0

    for pred, lab in zip(predictions, labels):
        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                total += 1
                if p_val == l_val:
                    correct += 1

    return {"accuracy": correct / total}

In [ ]:
# ==============================
# STEP 10: TRAINING CONFIG
# ==============================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    do_eval=True, # ✅ replaces evaluation_strategy
    eval_steps=200,
    logging_steps=100,           # ✅ replaces logging_strategy
    save_steps=500,              # ✅ replaces save_strategy
    save_total_limit=1
)

In [ ]:
# ==============================
# STEP 11: TRAINER
# ==============================

from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
# ==============================
# STEP 12: TRAIN
# ==============================

trainer.train()



In [ ]:
print(trainer.evaluate())

In [ ]:
# ==============================
# STEP 13: INFERENCE
# ==============================

def predict(sentence):
    inputs = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True).to(model.device)
    outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)

    result = []
    for word, pred in zip(sentence.split(), preds[0][1:len(sentence.split())+1]):
        result.append((word, id2label[pred.item()]))

    return result

In [ ]:
# ==============================
# STEP 14: TEST
# ==============================

print(predict("Elon Musk founded Tesla in California"))